# 《深山传家》Colab A100 批量 2.5D 粗墨线双光影烘培

用法：
1. 在 Google 云端硬盘新建 `shenshan_td/`，放入 `assets_manifest.json`、`bake_isometric_sprites.py`，并将生成的 `.glb` 模型放进 `shenshan_td/raw_models/`。
2. 菜单「代码执行程序 → 更改运行时类型」选 **GPU（T4 / L4 / A100 均可，无需 High-RAM）**。
3. 先将 `TEST_SINGLE` 设为单个文件名（如 `house_lv2_tile.glb`）验看画风与墨线粗细，满意后留空批量烘培全部模型。
4. 最后一格自动释放运行时，不浪费 Colab 单位。

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/shenshan_td"
BLENDER = "5.0.1"          # 与你现有 Blender 管线版本保持一致
TEST_SINGLE = ""           # 填文件名如 "house_lv2_tile.glb" 只烘 1 个；留空烘培全部
RES = 768                  # 精灵图分辨率（试看可改 512）
SAMPLES = 48               # Cycles 采样数（试看可改 24）
INK_WIDTH = 2.6            # Freestyle 粗墨线宽度（像素）
DEVICE = "auto"            # OPTIX 报错则改为 "CUDA"
OVERWRITE = False          # True 会重烘已有文件

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')

assert os.path.exists(f"{DRIVE_DIR}/bake_isometric_sprites.py"), "请先上传 bake_isometric_sprites.py"
assert os.path.exists(f"{DRIVE_DIR}/raw_models"), "请先建立 raw_models 目录并放入 .glb 模型"
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout)

mm = ".".join(BLENDER.split(".")[:2])
if not os.path.exists("/content/blender/blender"):
    !apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libxkbcommon0 libsm6 libgl1 libegl1 > /dev/null
    !wget -q https://download.blender.org/release/Blender{mm}/blender-{BLENDER}-linux-x64.tar.xz -O /content/blender.tar.xz
    !mkdir -p /content/blender && tar -xf /content/blender.tar.xz -C /content/blender --strip-components=1
!/content/blender/blender --version | head -1

In [ ]:
in_target = f"{DRIVE_DIR}/raw_models/{TEST_SINGLE}" if TEST_SINGLE else f"{DRIVE_DIR}/raw_models"
out_dir = f"{DRIVE_DIR}/baked_sprites"
manifest = f"{DRIVE_DIR}/assets_manifest.json"
ow = "--overwrite" if OVERWRITE else ""

!/content/blender/blender -b -P "{DRIVE_DIR}/bake_isometric_sprites.py" -- \
  --input "{in_target}" \
  --out "{out_dir}" \
  --manifest "{manifest}" \
  --res {RES} --samples {SAMPLES} --ink-width {INK_WIDTH} --device {DEVICE} {ow} \
  2>&1 | grep -E "^BAKE|^START|^BAKED|^SKIP|^ALL_|Error|Traceback"

In [ ]:
# 烘培完毕自动释放 Colab GPU 运行时，防止空转扣费
from google.colab import runtime
runtime.unassign()